# Автоматическая обработка данных такси

## Описание проекта

Каждый день компания обрабатывает миллионы поездок, оплаченных разными способами. Чтобы финансовая и продуктовая команды регулярно получали актуальные данные о выручке и поведении пассажиров, коллеги решили настроить автоматическую обработку этих данных.
#### Цель
Cоздать таблицу, которая станет основой для финансовых отчётов и аналитических дашбордов.
#### Задача 
Построить витрину данных, которая будет агрегировать информацию о поездках на такси по каждому способу оплаты. Для этого нужно написать PySpark-скрипт, который рассчитает ключевые показатели: количество поездок, среднюю стоимость поездки, средние чаевые и суммарную выручку по каждому типу оплаты.
#### Реализация 
Создать DAG в Airflow, который ежедневно:

* проверяет наличие новых файлов с данными;

* запускает Spark-задачу;

* формирует обновлённую итоговую таблицу.

## Описание данных

Таблица `taxi_data` содержит данные об активности пользователей и состоит из следующих полей:

* `taxi_id` — идентификатор водителя;

* `trip_start_timestamp` — время начала поездки;

* `trip_end_timestamp` — время окончания поездки;

* `trip_seconds` — длительность поездки в секундах;

* `trip_miles` — дистанция поездки;

* `fare` — стоимость поездки;

* `tips` — размер чаевых;

* `trip_total` — общая стоимость поездки: стоимость поездки + чаевые + комиссия;

* `payment_type` — способ оплаты.

## Пошаговая реализация - необходимо автоматизировать подготовку витрины данных по поездкам Такси:

1. Сначала напишу Spark-скрипт, который будет обрабатывать данные о поездках и агрегировать показатели по способам оплаты `payment_type`. Для этого рассчитываю несколько показателей:

* количество поездок, которое показывает общий спрос и загрузку сервиса;
* среднюю стоимость `fare`, которое отражает уровень среднего чека поездки;
* средние чаевые `tips` — индикатор удовлетворённости клиентов и мотивации водителей;
* суммарную выручку `trip_total` — ключевой показатель дохода компании.

! Все результаты будут собираться в одну итоговую таблицу `taxi_payment_summary`. После этого таблицу запишу в ClickHouse с помощью JDBC-драйвера.

2. Далее настрою DAG в Airflow, который будет запускаться ежедневно. Перед запуском он проверяет наличие файла с данными за нужную дату в S3-хранилище и только после появления файла запускает Spark-задачу.


## Шаг 1. Настройка Spark-агрегации

Данные хранятся в формате Parquet, поэтому для чтения использую метод `spark.read.parquet()`. Это быстрее и надёжнее, чем CSV.

Сгруппирую данные по полю `payment_type` и рассчитаю четыре показателя:

* количество поездок — `count(*)`;
* среднюю стоимость — `avg(...)`;
* средние чаевые — `avg(...)`;
* суммарную выручку — `sum(...)`.

Так получится витрина для анализа информации по каждому способу оплаты. После этого настрою запись полученной таблицы в ClickHouse. 

In [ ]:
# Импорт необходимых библиотек для работы с Spark и функциями
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys

# Создание Spark-сессии с названием и конфигурацией для S3 (Yandex Cloud Storage)
spark = SparkSession.builder \
    .appName("myAggregateTest") \
    .config("fs.s3a.endpoint", "storage.yandexcloud.net") \
    .getOrCreate()

# Настройка параметров подключения к базе данных ClickHouse
jdbcPort = 8443  # порт соединения
jdbcHostname = "rc1a-3jouval14nne7aun.mdb.yandexcloud.net"  # хостнейм базы данных
username = "da_20260106_ea7ec52eee"  # имя пользователя
jdbcDatabase = "playground_" + username  # имя базы данных
jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"  # URL подключения

# Чтение данных из parquet файла в S3-хранилище
taxiData = spark.read.parquet(f"s3a://da-plus-dags/project_04/taxi_data.parquet")

# Агрегация данных по типу оплаты с подсчётом, средним значением и суммой
result_df = taxiData.groupBy("payment_type").agg(
    F.count("*").alias("trip_count"),  # подсчёт количества поездок
    F.avg("fare").alias("avg_fare"),   # средняя цена проезда
    F.avg("tips").alias("avg_tips"),   # средний размер чаевых
    F.sum("trip_total").alias("total_revenue")  # общая выручка
)

# Записываем результат в ClickHouse
result_df.write.format("jdbc") \  # Настройка формата для записи через JDBC
    .option("url", jdbcUrl) \  # Установка URL соединения с базой данных
    .option("user", username) \  # Указание имени пользователя
    .option("password", "f81df282a24d4202bc0aa47c6cfd5326") \  # Пароль для доступа
    .option("dbtable", "taxi_payment_summary") \  # Таблица в базе данных, куда пишем
    .mode('append') \  # Режим добавления данных к существующим
    .save()  # Выполнение операции записи

## Шаг 2. Настройка DAG

В DAG использую `S3KeySensor`, чтобы дождаться появления файла в S3. После этого запускаю `DataprocCreatePysparkJobOperator`, передав путь к своему скрипту. 

Данные для подключения к Airflow:
*   IP — 89.169.152.149
*   Имя пользователя — da_20260106_ea7ec52eee
*   Пароль — f81df282a24d4202bc0aa47c6cfd5326

In [ ]:
# Задаем структуру и настройки DAG в Apache Airflow для автоматического ожидания файла в S3 и запуска PySpark задания
# filename=taxi_payment_summary_dag.py

from datetime import datetime, timedelta
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

# Создаём класс-наследник для оператора PySpark (может быть для расширения, если нужно)
class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id",)  # указываем поля, которые можно параметризовать шаблонами

# Название DAG
DAG_ID = "taxi_payment_summary_daily"

# Создаём DAG с расписанием и настройками
with DAG(
    DAG_ID,
    schedule_interval='@daily',  # запуск ежедневно
    start_date=datetime(2025, 1, 1),  # дата начала
    catchup=False  # не запускать пропущенные дни
) as dag:
    # Сенсор, ждет появления файла в S3
    wait_for_taxi_data = S3KeySensor(
        task_id='wait_for_taxi_data',  # идентификатор задачи
        bucket_name='da-plus-dags',  # имя бакета S3
        bucket_key='project_04/taxi_data.parquet',  # ключ файла
        poke_interval=300,  # интервал опроса — 5 минут
        timeout=60 * 60,  # таймаут — 1 час
        mode='poke',  # режим работы сенсора
        aws_conn_id='s3',  # подключение AWS
        soft_fail=False  # завершать с ошибкой, если файл не появился
    )

    # Запуск PySpark задания через Dataproc
    run_spark_job = DataprocCreatePysparkJobOperator(
        task_id='run_spark_job',  # идентификатор задачи
        cluster_id='c9q4134h5vi546h1e148',  # кластер Dataproc
        name='taxi_payment_summary',  # имя задания
        main_python_file_uri=f"s3a://da-plus-dags/da_20260106_ea7ec52eee/jobs/my_spark_job.py"  # путь к скрипту PySpark
    )

    # Задачи связаны: сначала ждем файл, затем запускаем Spark
    wait_for_taxi_data >> run_spark_job

## Шаг 3. Запуск DAG с помощью Airflow UI



Ссылка на скриншоты запуска: https://disk.yandex.ru/d/fcnlyVAhCz0HbA